In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/train.csv
/kaggle/input/competitions/nlp-getting-started/test.csv


In [2]:
df = pd.read_csv(r"/kaggle/input/competitions/nlp-getting-started/train.csv")
df.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [3]:
df.tail(5)

,id,keyword,location,text,target
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1
7612,10873,NaN,NaN,The Latest: More Homes Razed by Northern Calif...,1


In [4]:
df.shape

(7613, 5)

In [5]:
df['text'].nunique()

7503

In [6]:
df.loc[df['id'] == 6094, 'text']

4290    #Allah describes piling up #wealth thinking it...
Name: text, dtype: object

In [7]:
print(df.loc[df['id'] == 6094, 'text'].iloc[0])

#Allah describes piling up #wealth thinking it would last #forever as the description of the people of #Hellfire in Surah Humaza. #Reflect


In [8]:
duplicates = df[df['text'].duplicated(keep=False)].sort_values('text')

duplicates[['id', 'text', 'target']].head(5)

,id,text,target
4290,6094,#Allah describes piling up #wealth thinking it...,0
4299,6105,#Allah describes piling up #wealth thinking it...,0
4312,6123,#Allah describes piling up #wealth thinking it...,1
6363,9095,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6373,9107,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1


In [9]:
df.isnull().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [10]:
df.drop(columns = ['keyword', 'location',], inplace = True)
df

,id,text,target
0,1,Our Deeds are the Reason of this #earthquake M...,1
1,4,Forest fire near La Ronge Sask. Canada,1
2,5,All residents asked to 'shelter in place' are ...,1
3,6,"13,000 people receive #wildfires evacuation or...",1
4,7,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...
7608,10869,Two giant cranes holding a bridge collapse int...,1
7609,10870,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,Police investigating after an e-bike collided ...,1


In [11]:
import re
import nltk 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = text.lower()
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'[^a-z0-9#_ ]+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]','',text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)
df['preprocessed_text'] = df['text'].apply(preprocess)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [13]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)

In [14]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [15]:
X = vectorizer.fit_transform(df['preprocessed_text'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)
log = LogisticRegression(max_iter = 1000, random_state = 42)
svm_model = SVC(kernel = 'linear', random_state = 42)

model = svm_model.fit(X_train, y_train)
modell = log.fit(X_train, y_train)

In [16]:
predict = modell.predict(X_test)
predictt = model.predict(X_test)

In [17]:
from sklearn.metrics import accuracy_score, f1_score

print(accuracy_score(y_test, predict))
print(accuracy_score(y_test, predictt))

0.8003939592908733
0.7944845699277742


In [18]:
from sklearn.ensemble import GradientBoostingClassifier

In [19]:
gradd = GradientBoostingClassifier(random_state=42)
grd = grd.fit(X_train, y_train)
predicttt = grd.predict(X_test)

NameError: name 'grd' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print(accuracy_score(y_test, predicttt))
print(f1_score(y_test, predicttt))


In [ ]:
from sklearn.linear_model import SGDClassifier

sgd=SGDClassifier(
    loss="log_loss", #logistic regression
    max_iter=1000, #maximum number of iterations
    learning_rate='optimal',
    random_state=42
)

sgd.fit(X_train,y_train)

In [ ]:

y_pred=sgd.predict(X_test)
print(y_pred)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import pandas as pd

# Convert TF-IDF sparse matrix to dense
X_dense = X.toarray()

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "SVM": SVC(
        kernel="linear"
    ),

    "KNN": KNeighborsClassifier(),

    "Naive Bayes": GaussianNB(),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

results = []

for name, model in models.items():

    print("Training:", name)

    model.fit(X_dense, y)

    pred = model.predict(X_dense)

    acc = accuracy_score(y, pred)

    results.append({
        "Model": name,
        "Accuracy": acc
    })

results = pd.DataFrame(results)

results = results.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results

In [ ]:
from sklearn.metrics import *

print("Accuracy:", accuracy_score(y_test,y_pred))

distilroberta

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")

df.head()

In [ ]:
print(df.shape)
print(df['target'].value_counts())

In [ ]:
df = df[['text', 'target']].copy()

df = df.dropna(subset=['text'])

df.head()

In [ ]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[['text', 'target']],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[['text', 'target']],
    preserve_index=False
)

In [ ]:
train_dataset = train_dataset.rename_column("target", "labels")
valid_dataset = valid_dataset.rename_column("target", "labels")

In [ ]:
model_name = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

valid_tokenized = valid_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
train_tokenized = train_tokenized.remove_columns(["text"])
valid_tokenized = valid_tokenized.remove_columns(["text"])

In [ ]:
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilroberta-disaster",
    
    eval_strategy="epoch",
    save_strategy="epoch",
    
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    num_train_epochs=3,
    
    weight_decay=0.01,
    
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_steps=50,
    
    report_to="none"
)

but got worst results using learning rate 1e-5 and epochs = 5

In [ ]:
# training_args = TrainingArguments(
#     output_dir="./distilroberta-disaster",
    
#     eval_strategy="epoch",
#     save_strategy="epoch",
    
#     learning_rate=1e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
    
#     num_train_epochs=5,
    
#     weight_decay=0.01,
    
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     greater_is_better=True,
    
#     logging_steps=50,
    
#     report_to="none"
# )

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [ ]:
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

results

In [ ]:
print("Accuracy :", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall   :", results["eval_recall"])
print("F1       :", results["eval_f1"])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred_output = trainer.predict(valid_tokenized)

predictions = np.argmax(
    pred_output.predictions,
    axis=1
)

true_labels = pred_output.label_ids

print(confusion_matrix(true_labels, predictions))

print(
    classification_report(
        true_labels,
        predictions,
        target_names=["Not Disaster", "Disaster"]
    )
)

In [ ]:
full_dataset = Dataset.from_pandas(
    df[['text', 'target']],
    preserve_index=False
)

full_dataset = full_dataset.rename_column(
    "target",
    "labels"
)

In [ ]:
full_tokenized = full_dataset.map(
    tokenize_function,
    batched=True
)

full_tokenized = full_tokenized.remove_columns(["text"])

In [ ]:
final_training_args = TrainingArguments(
    output_dir="./distilroberta-final",
    
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    
    num_train_epochs=3,
    
    weight_decay=0.01,
    
    logging_steps=50,
    
    report_to="none"
)

but got worst results using learning rate 1e-5 and epochs = 5

In [ ]:
# final_training_args = TrainingArguments(
#     output_dir="./distilroberta-final",
    
#     learning_rate=1e-5,
#     per_device_train_batch_size=16,
    
#     num_train_epochs=5,
    
#     weight_decay=0.01,
    
#     logging_steps=50,
    
#     report_to="none"
# )

In [ ]:
final_trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=full_tokenized,
    processing_class=tokenizer
)

In [ ]:
final_trainer.train()

In [ ]:
test = pd.read_csv(
    "/kaggle/input/competitions/nlp-getting-started/test.csv"
)

test.head()

In [ ]:
test_dataset = Dataset.from_pandas(
    test[['text']],
    preserve_index=False
)

In [ ]:
test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
test_tokenized = test_tokenized.remove_columns(["text"])

In [ ]:
test_output = final_trainer.predict(test_tokenized)

test_predictions = np.argmax(
    test_output.predictions,
    axis=1
)

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": test_predictions
})

submission.head()

In [ ]:
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.shape)
print(submission.head(10))

In [ ]:
import os

print(os.path.exists("/kaggle/working/submission.csv"))

In [ ]:
print(submission.shape)
print(submission.head(10))

In [ ]:
import os

path = "/kaggle/working/submission.csv"

print(os.path.exists(path))
print(os.path.getsize(path))

In [ ]:
submission_check = pd.read_csv("/kaggle/working/submission.csv")

print(submission_check.shape)
print(submission_check.columns.tolist())
print(submission_check.head())

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/submission.csv")

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/submission.csv")

In [ ]:
print(submission.shape)
print(submission.columns)

with cross validation

In [ ]:
"""
Disaster Tweets Classification - DistilRoBERTa (fixed pipeline)
=================================================================
Fixes applied vs the original notebook:
1. Fresh model reload before every training run (no accidental continued
   fine-tuning on top of a previously trained model).
2. compute_metrics defined BEFORE any Trainer is built.
3. seed fixed everywhere for reproducible comparisons.
4. Learning rate set to 2e-5 (standard fine-tuning range), not 1e-5 or 2e-3.
5. EarlyStoppingCallback actually attached to the Trainer that trains.
6. Duplicate texts with conflicting labels are dropped before training.
7. Light, transformer-appropriate text cleaning (URL/user normalization only
   -- no stopword removal / lemmatization, which hurts transformers).
8. gradient_accumulation_steps added for a larger effective batch size.
9. Final full-data retrain uses a fresh model too, and test predictions are
   generated from that fresh full-data model.
"""

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ---------------------------------------------------------------------------
# 0. Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 128
TRAIN_CSV = "/kaggle/input/competitions/nlp-getting-started/train.csv"
TEST_CSV = "/kaggle/input/competitions/nlp-getting-started/test.csv"
OUTPUT_DIR = "/kaggle/working"

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(TRAIN_CSV)
df = df[["text", "target"]].dropna(subset=["text"]).reset_index(drop=True)

print("Shape before dedup:", df.shape)
print(df["target"].value_counts())

# ---------------------------------------------------------------------------
# 2. Drop duplicate texts with CONFLICTING labels (label noise)
#    (keep duplicates that agree on label -- just drop the noisy ones)
# ---------------------------------------------------------------------------
dupe_mask = df.duplicated(subset="text", keep=False)
dupes = df[dupe_mask]
conflicting_texts = (
    dupes.groupby("text")["target"].nunique()
    .loc[lambda s: s > 1]
    .index
)
n_conflicting = df["text"].isin(conflicting_texts).sum()
print(f"Dropping {n_conflicting} rows with label-conflicting duplicate text")
df = df[~df["text"].isin(conflicting_texts)].reset_index(drop=True)

# also drop exact duplicate rows (same text + same label) to avoid leakage
# across train/val split
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("Shape after dedup:", df.shape)

# ---------------------------------------------------------------------------
# 3. Light, transformer-appropriate cleaning
#    (NOT stopword removal / lemmatization -- that hurts transformers)
# ---------------------------------------------------------------------------
def light_clean(text: str) -> str:
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)  # collapse "soooooo" -> "sooo"
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(light_clean)

# ---------------------------------------------------------------------------
# 4. Train/val split
# ---------------------------------------------------------------------------
train_df, valid_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["target"]
)
print("Train:", train_df.shape, "Validation:", valid_df.shape)

train_dataset = Dataset.from_pandas(train_df[["text", "target"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df[["text", "target"]], preserve_index=False)

train_dataset = train_dataset.rename_column("target", "labels")
valid_dataset = valid_dataset.rename_column("target", "labels")

# ---------------------------------------------------------------------------
# 5. Tokenizer / tokenization
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"])
valid_tokenized = valid_tokenized.remove_columns(["text"])

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ---------------------------------------------------------------------------
# 6. compute_metrics defined BEFORE any Trainer (fixes the NameError bug)
# ---------------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

# ---------------------------------------------------------------------------
# 7. Helper: always load a FRESH model (fixes the "continued training" bug)
# ---------------------------------------------------------------------------
def fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

# ---------------------------------------------------------------------------
# 8. Train/validate with a clean, standard config
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/distilroberta-run",
    seed=SEED,

    num_train_epochs=4,                 # early stopping will cut this short if needed
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,      # effective batch size = 32

    learning_rate=2e-5,                 # standard fine-tuning LR (not 1e-5, not 2e-3)
    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
)

model = fresh_model()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # actually attached now
)

trainer.train()

results = trainer.evaluate()
print("Accuracy :", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall   :", results["eval_recall"])
print("F1       :", results["eval_f1"])

pred_output = trainer.predict(valid_tokenized)
predictions = np.argmax(pred_output.predictions, axis=1)
true_labels = pred_output.label_ids

print(confusion_matrix(true_labels, predictions))
print(classification_report(true_labels, predictions, target_names=["Not Disaster", "Disaster"]))

# ---------------------------------------------------------------------------
# 9. Final retrain on full data with a FRESH model (not the validated one)
# ---------------------------------------------------------------------------
full_df = df.copy()
full_dataset = Dataset.from_pandas(full_df[["text", "target"]], preserve_index=False)
full_dataset = full_dataset.rename_column("target", "labels")

full_tokenized = full_dataset.map(tokenize_function, batched=True)
full_tokenized = full_tokenized.remove_columns(["text"])

# Use the epoch count that gave the best validation F1 in the run above
# (rounded up), so the full-data retrain trains for a comparable amount --
# early stopping isn't meaningful here since there's no held-out val set.
best_epoch = max(1, round(trainer.state.epoch or 4))

final_training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/distilroberta-final",
    seed=SEED,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,

    num_train_epochs=best_epoch,
    weight_decay=0.01,
    warmup_ratio=0.1,

    logging_steps=50,
    report_to="none",
)

final_model = fresh_model()  # fresh, not the already-fine-tuned validation model

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tokenized,
    processing_class=tokenizer,
)

final_trainer.train()

# ---------------------------------------------------------------------------
# 10. Predict on test set and write submission
# ---------------------------------------------------------------------------
test = pd.read_csv(TEST_CSV)
test["text"] = test["text"].apply(light_clean)

test_dataset = Dataset.from_pandas(test[["text"]], preserve_index=False)
test_tokenized = test_dataset.map(tokenize_function, batched=True)
test_tokenized = test_tokenized.remove_columns(["text"])

test_output = final_trainer.predict(test_tokenized)
test_predictions = np.argmax(test_output.predictions, axis=1)

submission = pd.DataFrame({"id": test["id"], "target": test_predictions})
submission_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.shape)
print(submission.head(10))

roberat-base

In [ ]:
"""
Disaster Tweets Classification - DistilRoBERTa (fixed pipeline)
=================================================================
Fixes applied vs the original notebook:
1. Fresh model reload before every training run (no accidental continued
   fine-tuning on top of a previously trained model).
2. compute_metrics defined BEFORE any Trainer is built.
3. seed fixed everywhere for reproducible comparisons.
4. Learning rate set to 2e-5 (standard fine-tuning range), not 1e-5 or 2e-3.
5. EarlyStoppingCallback actually attached to the Trainer that trains.
6. Duplicate texts with conflicting labels are dropped before training.
7. Light, transformer-appropriate text cleaning (URL/user normalization only
   -- no stopword removal / lemmatization, which hurts transformers).
8. gradient_accumulation_steps added for a larger effective batch size.
9. Final full-data retrain uses a fresh model too, and test predictions are
   generated from that fresh full-data model.
10. Optional Stratified K-Fold cross-validation (set RUN_CV = True) to get
    a trustworthy mean +/- std accuracy instead of trusting a single split.
"""

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ---------------------------------------------------------------------------
# 0. Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "roberta-base"   # was distilroberta-base -- bigger model, more capacity
MAX_LENGTH = 128
NUM_EPOCHS = 4
N_FOLDS = 5
RUN_CV = True   # <-- set False to skip CV and just do a single 80/20 split
TRAIN_CSV = "/kaggle/input/competitions/nlp-getting-started/train.csv"
TEST_CSV = "/kaggle/input/competitions/nlp-getting-started/test.csv"
OUTPUT_DIR = "/kaggle/working"

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(TRAIN_CSV)
df = df[["text", "target"]].dropna(subset=["text"]).reset_index(drop=True)

print("Shape before dedup:", df.shape)
print(df["target"].value_counts())

# ---------------------------------------------------------------------------
# 2. Drop duplicate texts with CONFLICTING labels (label noise)
#    (keep duplicates that agree on label -- just drop the noisy ones)
# ---------------------------------------------------------------------------
dupe_mask = df.duplicated(subset="text", keep=False)
dupes = df[dupe_mask]
conflicting_texts = (
    dupes.groupby("text")["target"].nunique()
    .loc[lambda s: s > 1]
    .index
)
n_conflicting = df["text"].isin(conflicting_texts).sum()
print(f"Dropping {n_conflicting} rows with label-conflicting duplicate text")
df = df[~df["text"].isin(conflicting_texts)].reset_index(drop=True)

# also drop exact duplicate rows (same text + same label) to avoid leakage
# across train/val split
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("Shape after dedup:", df.shape)

# ---------------------------------------------------------------------------
# 3. Light, transformer-appropriate cleaning
#    (NOT stopword removal / lemmatization -- that hurts transformers)
# ---------------------------------------------------------------------------
def light_clean(text: str) -> str:
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)  # collapse "soooooo" -> "sooo"
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(light_clean)

# ---------------------------------------------------------------------------
# 4. Train/val split
# ---------------------------------------------------------------------------
train_df, valid_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["target"]
)
print("Train:", train_df.shape, "Validation:", valid_df.shape)

train_dataset = Dataset.from_pandas(train_df[["text", "target"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df[["text", "target"]], preserve_index=False)

train_dataset = train_dataset.rename_column("target", "labels")
valid_dataset = valid_dataset.rename_column("target", "labels")

# ---------------------------------------------------------------------------
# 5. Tokenizer / tokenization
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"])
valid_tokenized = valid_tokenized.remove_columns(["text"])

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ---------------------------------------------------------------------------
# 6. compute_metrics defined BEFORE any Trainer (fixes the NameError bug)
# ---------------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

# ---------------------------------------------------------------------------
# 7. Helper: always load a FRESH model (fixes the "continued training" bug)
# ---------------------------------------------------------------------------
def fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

# ---------------------------------------------------------------------------
# 8. Train/validate with a clean, standard config
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/distilroberta-run",
    seed=SEED,

    num_train_epochs=NUM_EPOCHS,        # early stopping will cut this short if needed
    per_device_train_batch_size=8,      # lowered from 16 -- roberta-base uses ~1.5x the memory
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,      # effective batch size still = 32

    learning_rate=2e-5,                 # standard fine-tuning LR (not 1e-5, not 2e-3)
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),     # faster + less memory on GPU, no effect on CPU

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
)

model = fresh_model()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # actually attached now
)

trainer.train()

results = trainer.evaluate()
print("Accuracy :", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall   :", results["eval_recall"])
print("F1       :", results["eval_f1"])

pred_output = trainer.predict(valid_tokenized)
predictions = np.argmax(pred_output.predictions, axis=1)
true_labels = pred_output.label_ids

print(confusion_matrix(true_labels, predictions))
print(classification_report(true_labels, predictions, target_names=["Not Disaster", "Disaster"]))

# ---------------------------------------------------------------------------
# 9. Final retrain on full data with a FRESH model (not the validated one)
# ---------------------------------------------------------------------------
full_df = df.copy()
full_dataset = Dataset.from_pandas(full_df[["text", "target"]], preserve_index=False)
full_dataset = full_dataset.rename_column("target", "labels")

full_tokenized = full_dataset.map(tokenize_function, batched=True)
full_tokenized = full_tokenized.remove_columns(["text"])

# Use the epoch count that gave the best validation F1 in the run above
# (rounded up), so the full-data retrain trains for a comparable amount --
# early stopping isn't meaningful here since there's no held-out val set.
best_epoch = max(1, round(trainer.state.epoch or 4))

final_training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/distilroberta-final",
    seed=SEED,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,

    num_train_epochs=best_epoch,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),

    logging_steps=50,
    report_to="none",
)

final_model = fresh_model()  # fresh, not the already-fine-tuned validation model

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tokenized,
    processing_class=tokenizer,
)

final_trainer.train()

# ---------------------------------------------------------------------------
# 10. Predict on test set and write submission
# ---------------------------------------------------------------------------
test = pd.read_csv(TEST_CSV)
test["text"] = test["text"].apply(light_clean)

test_dataset = Dataset.from_pandas(test[["text"]], preserve_index=False)
test_tokenized = test_dataset.map(tokenize_function, batched=True)
test_tokenized = test_tokenized.remove_columns(["text"])

test_output = final_trainer.predict(test_tokenized)
test_predictions = np.argmax(test_output.predictions, axis=1)

submission = pd.DataFrame({"id": test["id"], "target": test_predictions})
submission_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.shape)
print(submission.head(10))

BERTWEET

In [ ]:
"""
Disaster Tweets Classification - BERTweet (vinai/bertweet-base)
=================================================================
Same fixes as the distilroberta/roberta scripts, plus BERTweet-specific
handling:
1. Fresh model reload before every training run (no accidental continued
   fine-tuning on top of a previously trained model).
2. compute_metrics defined BEFORE any Trainer is built.
3. seed fixed everywhere for reproducible comparisons.
4. Learning rate 2e-5, standard fine-tuning range.
5. EarlyStoppingCallback actually attached to the Trainer that trains.
6. Duplicate texts with conflicting labels are dropped before training.
7. BERTweet's own normalizer handles URL/@mention/emoji normalization
   (it converts them to "HTTPURL" / "@USER" and demojizes emoji) -- do NOT
   run the generic light_clean regex from the other scripts, it would fight
   with BERTweet's normalizer and hurt results. We only collapse repeated
   characters ourselves, then let normalization=True do the rest.
8. gradient_accumulation_steps for a larger effective batch size.
9. Final full-data retrain uses a fresh model too.
10. Optional Stratified K-Fold cross-validation (set RUN_CV = True).

IMPORTANT setup step (run once): BERTweet's normalizer needs the `emoji`
package to demojize emoji correctly:
    !pip install -q emoji==0.6.0
(emoji must be pinned to 0.6.0 -- newer emoji versions changed their API
and break BERTweet's normalizer.)
"""

# Run once before this script, in its own cell:
#   !pip install -q emoji==0.6.0
# BERTweet's normalizer needs this exact old version of `emoji`; newer
# versions changed their API and will raise an error inside the tokenizer.

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ---------------------------------------------------------------------------
# 0. Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "vinai/bertweet-base"  # tweet-domain pretraining, not just bigger
MAX_LENGTH = 128   # matches BERTweet's own pretraining max sequence length
NUM_EPOCHS = 4
N_FOLDS = 5
RUN_CV = True   # <-- set False to skip CV and just do a single 80/20 split
TRAIN_CSV = "/kaggle/input/competitions/nlp-getting-started/train.csv"
TEST_CSV = "/kaggle/input/competitions/nlp-getting-started/test.csv"
OUTPUT_DIR = "/kaggle/working"

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(TRAIN_CSV)
df = df[["text", "target"]].dropna(subset=["text"]).reset_index(drop=True)

print("Shape before dedup:", df.shape)
print(df["target"].value_counts())

# ---------------------------------------------------------------------------
# 2. Drop duplicate texts with CONFLICTING labels (label noise)
#    (keep duplicates that agree on label -- just drop the noisy ones)
# ---------------------------------------------------------------------------
dupe_mask = df.duplicated(subset="text", keep=False)
dupes = df[dupe_mask]
conflicting_texts = (
    dupes.groupby("text")["target"].nunique()
    .loc[lambda s: s > 1]
    .index
)
n_conflicting = df["text"].isin(conflicting_texts).sum()
print(f"Dropping {n_conflicting} rows with label-conflicting duplicate text")
df = df[~df["text"].isin(conflicting_texts)].reset_index(drop=True)

# also drop exact duplicate rows (same text + same label) to avoid leakage
# across train/val split
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("Shape after dedup:", df.shape)

# ---------------------------------------------------------------------------
# 3. Minimal cleaning only -- BERTweet's tokenizer normalization (enabled
#    below) already converts URLs -> "HTTPURL", @mentions -> "@USER", and
#    demojizes emoji. Doing that ourselves first would double up / conflict
#    with it, so we only collapse repeated characters here.
# ---------------------------------------------------------------------------
def light_clean(text: str) -> str:
    text = str(text)
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)  # collapse "soooooo" -> "sooo"
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(light_clean)

# ---------------------------------------------------------------------------
# 4. Train/val split
# ---------------------------------------------------------------------------
train_df, valid_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["target"]
)
print("Train:", train_df.shape, "Validation:", valid_df.shape)

train_dataset = Dataset.from_pandas(train_df[["text", "target"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df[["text", "target"]], preserve_index=False)

train_dataset = train_dataset.rename_column("target", "labels")
valid_dataset = valid_dataset.rename_column("target", "labels")

# ---------------------------------------------------------------------------
# 5. Tokenizer / tokenization
#    normalization=True turns on BERTweet's tweet-specific text normalizer
#    (URL -> HTTPURL, @mention -> @USER, emoji -> demojized text). Requires
#    `emoji==0.6.0` installed (see module docstring).
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"])
valid_tokenized = valid_tokenized.remove_columns(["text"])

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# ---------------------------------------------------------------------------
# 6. compute_metrics defined BEFORE any Trainer (fixes the NameError bug)
# ---------------------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

# ---------------------------------------------------------------------------
# 7. Helper: always load a FRESH model (fixes the "continued training" bug)
# ---------------------------------------------------------------------------
def fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

# ---------------------------------------------------------------------------
# 8. Train/validate with a clean, standard config
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/bertweet-run",
    seed=SEED,

    num_train_epochs=NUM_EPOCHS,        # early stopping will cut this short if needed
    per_device_train_batch_size=8,      # lowered from 16 -- roberta-base uses ~1.5x the memory
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,      # effective batch size still = 32

    learning_rate=2e-5,                 # standard fine-tuning LR (not 1e-5, not 2e-3)
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),     # faster + less memory on GPU, no effect on CPU

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
)

model = fresh_model()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # actually attached now
)

trainer.train()

results = trainer.evaluate()
print("Accuracy :", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall   :", results["eval_recall"])
print("F1       :", results["eval_f1"])

pred_output = trainer.predict(valid_tokenized)
predictions = np.argmax(pred_output.predictions, axis=1)
true_labels = pred_output.label_ids

print(confusion_matrix(true_labels, predictions))
print(classification_report(true_labels, predictions, target_names=["Not Disaster", "Disaster"]))

# ---------------------------------------------------------------------------
# 9. Final retrain on full data with a FRESH model (not the validated one)
# ---------------------------------------------------------------------------
full_df = df.copy()
full_dataset = Dataset.from_pandas(full_df[["text", "target"]], preserve_index=False)
full_dataset = full_dataset.rename_column("target", "labels")

full_tokenized = full_dataset.map(tokenize_function, batched=True)
full_tokenized = full_tokenized.remove_columns(["text"])

# Use the epoch count that gave the best validation F1 in the run above
# (rounded up), so the full-data retrain trains for a comparable amount --
# early stopping isn't meaningful here since there's no held-out val set.
best_epoch = max(1, round(trainer.state.epoch or 4))

final_training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/bertweet-final",
    seed=SEED,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,

    num_train_epochs=best_epoch,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),

    logging_steps=50,
    report_to="none",
)

final_model = fresh_model()  # fresh, not the already-fine-tuned validation model

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tokenized,
    processing_class=tokenizer,
)

final_trainer.train()

# ---------------------------------------------------------------------------
# 10. Predict on test set and write submission
# ---------------------------------------------------------------------------
test = pd.read_csv(TEST_CSV)
test["text"] = test["text"].apply(light_clean)

test_dataset = Dataset.from_pandas(test[["text"]], preserve_index=False)
test_tokenized = test_dataset.map(tokenize_function, batched=True)
test_tokenized = test_tokenized.remove_columns(["text"])

test_output = final_trainer.predict(test_tokenized)
test_predictions = np.argmax(test_output.predictions, axis=1)

submission = pd.DataFrame({"id": test["id"], "target": test_predictions})
submission_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.shape)
print(submission.head(10))